# Part 2 — Exploratory Data Analysis

> **Work in progress:** This notebook is structurally complete and its plotting code is smoke-tested, but the project findings are not complete until Part 1 exports the canonical processed arrays and manifests. When those artifacts are missing, every figure is explicitly labeled as a synthetic smoke test and must not be quoted as a project result.

This notebook owns the final-project EDA: parameter distributions, target relationships, strain plots, and O3a/O4a comparisons. It builds on the data acquisition, injection, and preprocessing work already completed by the team without importing another notebook's implementation.

## TL;DR

- **Available now:** the Part 1 artifact contract, reusable validation and plotting functions, an inventory of the team's source notebooks, and deterministic synthetic smoke checks.
- **Still required:** `processed.npz`, `processed_manifest.csv`, `processing_settings.json`, and the corresponding simulation/data manifests for one agreed `experiment_id`.
- **Boundary:** this notebook describes the data. It does not train a model, calculate model feature importance, or make final posterior-calibration claims.
- **Execution rule:** rerun the notebook from the top after Part 1 artifacts arrive. Only the real-artifact branch may support project conclusions.

## Context and methods

The project asks whether neural posterior estimation trained with O3a H1 noise remains reliable when evaluated with O4a H1 noise. Part 2 examines whether the simulated target space, model-ready strain, and detector-noise summaries are suitable for that comparison.

### Key assumptions

1. A sample is identified by `sample_id`; tables are never joined by row position.
2. `group_id` is the leakage-control unit and may belong to only one split.
3. Target order is `chirp_mass`, `mass_ratio`, `chi_eff`, `distance`.
4. O4a is evaluation-only. It is not used to choose preprocessing, features, or model hyperparameters.
5. Processed-injection differences can be confounded by target values and SNR. Noise-only PSD/ASD comparisons are preferred for claims about detector drift.
6. Synthetic smoke data verify code paths only; they are not evidence about LIGO data.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal


def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ligo-parameter-inference repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
EXPERIMENT_ID = os.environ.get("LIGO_EXPERIMENT_ID", "manual_preprocessing_v1")
EXPERIMENT_ROOT = PROJECT_ROOT / "artifacts" / "manual_preprocessing_milestone" / EXPERIMENT_ID
DATA_DIR = EXPERIMENT_ROOT / "data"
SIMULATION_DIR = EXPERIMENT_ROOT / "simulation"
PREPROCESSING_DIR = EXPERIMENT_ROOT / "preprocessing"

TARGET_NAMES = ["chirp_mass", "mass_ratio", "chi_eff", "distance"]
TARGET_LABELS = {
    "chirp_mass": "Chirp mass (solar masses)",
    "mass_ratio": "Mass ratio q (dimensionless)",
    "chi_eff": "Effective spin (dimensionless)",
    "distance": "Luminosity distance (Mpc)",
}
RUN_COLORS = {"O3a": "#0072B2", "O4a": "#D55E00"}
SAMPLE_RATE_HZ = 4096
WINDOW_SECONDS = 4.0
RANDOM_SEED = 207

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("Project root:", PROJECT_ROOT)
print("Experiment ID:", EXPERIMENT_ID)
print("Experiment root:", EXPERIMENT_ROOT)

## Part 1 handoff contract

The analysis expects the experiment-specific files below. Core EDA uses `processed.npz`, `processed_manifest.csv`, `targets.csv`, and `processing_settings.json`. Noise-only O3a/O4a comparison additionally uses `noise_manifest.csv` and `preprocessing/psd/*.npz`.

The notebook never creates or modifies these inputs. Missing files remain visibly pending.

In [ ]:
ARTIFACT_PATHS = {
    "experiment settings": DATA_DIR / "experiment.json",
    "noise manifest": DATA_DIR / "noise_manifest.csv",
    "injection attempts": SIMULATION_DIR / "attempts.csv",
    "injection manifest": SIMULATION_DIR / "injection_manifest.csv",
    "targets": SIMULATION_DIR / "targets.csv",
    "simulation settings": SIMULATION_DIR / "simulation_settings.json",
    "processed arrays": PREPROCESSING_DIR / "processed.npz",
    "processed manifest": PREPROCESSING_DIR / "processed_manifest.csv",
    "processing settings": PREPROCESSING_DIR / "processing_settings.json",
    "rejection ledger": PREPROCESSING_DIR / "rejections.csv",
}

artifact_status = pd.DataFrame([
    {
        "artifact": name,
        "status": "available" if path.is_file() else "pending",
        "repo_relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
    }
    for name, path in ARTIFACT_PATHS.items()
])

CORE_INPUTS = [
    ARTIFACT_PATHS["processed arrays"],
    ARTIFACT_PATHS["processed manifest"],
    ARTIFACT_PATHS["processing settings"],
    ARTIFACT_PATHS["targets"],
]
REAL_DATA_READY = all(path.is_file() for path in CORE_INPUTS)

print(artifact_status.to_string(index=False))
print("\nCanonical Part 1 handoff ready:", REAL_DATA_READY)

## Existing team work used as context

The notebooks below remain the provenance for how the current contract was developed:

- Role 1: raw GWOSC acquisition, source groups, splits, and raw strain EDA.
- Role 2: waveform simulation, target construction, SNR selection, and injection manifests.
- Role 3 / PR #2: PSD estimation, whitening, filtering, cropping, QC, and processed-array export.
- Role 4: an earlier dataset audit and EDA synthesis for a small pilot.

They are inventoried here, but their code and hidden kernel state are not imported. The final EDA must use explicit Part 1 files so that it can be rerun independently.

In [ ]:
TEAM_NOTEBOOKS = {
    "raw data and provenance": PROJECT_ROOT / "notebooks/milestone_preprocessing_roles/01_raw_data_provenance_and_eda.ipynb",
    "simulations and labels": PROJECT_ROOT / "notebooks/milestone_preprocessing_roles/02_simulations_labels_and_selection.ipynb",
    "preprocessing and QC (PR #2)": PROJECT_ROOT / "notebooks/milestone_preprocessing_roles/03_preprocessing_and_quality_checks.ipynb",
    "earlier audit and EDA synthesis": PROJECT_ROOT / "notebooks/milestone_preprocessing_roles/04_dataset_audit_eda_and_methodology_plan.ipynb",
    "baseline EDA reference": PROJECT_ROOT / "notebooks/baseline_eda.ipynb",
}

inventory_rows = []
for role, path in TEAM_NOTEBOOKS.items():
    notebook = json.loads(path.read_text())
    code_cells = [cell for cell in notebook["cells"] if cell["cell_type"] == "code"]
    inventory_rows.append({
        "source": role,
        "path": path.relative_to(PROJECT_ROOT).as_posix(),
        "code_cells": len(code_cells),
        "nonempty_code_cells": sum(bool("".join(cell.get("source", "")).strip()) for cell in code_cells),
        "executed_cells": sum(cell.get("execution_count") is not None for cell in code_cells),
        "saved_outputs": sum(len(cell.get("outputs", [])) for cell in code_cells),
    })

team_notebook_inventory = pd.DataFrame(inventory_rows)
print(team_notebook_inventory.to_string(index=False))

## Load and validate the canonical dataset

This validation is intentionally strict. If Part 1 files exist but disagree, the notebook raises an error instead of silently switching to demonstration data.

In [ ]:
def require_columns(frame, required, name):
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")


def load_part1_handoff(experiment_root):
    preprocessing_dir = experiment_root / "preprocessing"
    simulation_dir = experiment_root / "simulation"

    manifest = pd.read_csv(preprocessing_dir / "processed_manifest.csv")
    targets = pd.read_csv(simulation_dir / "targets.csv")
    settings = json.loads((preprocessing_dir / "processing_settings.json").read_text())

    required_manifest = ["row_index", "sample_id", "noise_id", "group_id", "run", "split", "n_samples"]
    require_columns(manifest, required_manifest, "processed_manifest.csv")
    require_columns(targets, ["sample_id", *TARGET_NAMES], "targets.csv")

    if manifest["sample_id"].duplicated().any() or targets["sample_id"].duplicated().any():
        raise ValueError("sample_id must be unique in both processed and target manifests.")

    manifest = manifest.sort_values("row_index").reset_index(drop=True)
    if not np.array_equal(manifest["row_index"].to_numpy(), np.arange(len(manifest))):
        raise ValueError("row_index must be contiguous from zero and define processed-array order.")

    with np.load(preprocessing_dir / "processed.npz", allow_pickle=False) as arrays:
        required_keys = {"X", "theta", "sample_id", "split", "noise_id", "group_id", "run"}
        missing_keys = required_keys - set(arrays.files)
        if missing_keys:
            raise ValueError(f"processed.npz is missing keys: {sorted(missing_keys)}")
        X = np.asarray(arrays["X"], dtype=np.float64)
        theta = np.asarray(arrays["theta"], dtype=np.float64)
        array_ids = arrays["sample_id"].astype(str)
        array_split = arrays["split"].astype(str)
        array_run = arrays["run"].astype(str)
        array_group = arrays["group_id"].astype(str)

    if X.ndim != 2 or theta.shape != (len(X), len(TARGET_NAMES)):
        raise ValueError(f"Unexpected array shapes: X={X.shape}, theta={theta.shape}")
    if len(manifest) != len(X):
        raise ValueError("processed manifest and X have different row counts.")
    if not np.isfinite(X).all() or not np.isfinite(theta).all():
        raise ValueError("Processed strain or targets contain nonfinite values.")
    if not np.array_equal(array_ids, manifest["sample_id"].astype(str).to_numpy()):
        raise ValueError("processed.npz sample_id order does not match processed_manifest.csv.")
    if not np.array_equal(array_split, manifest["split"].astype(str).to_numpy()):
        raise ValueError("Split values disagree between processed arrays and manifest.")
    if not np.array_equal(array_run, manifest["run"].astype(str).to_numpy()):
        raise ValueError("Run values disagree between processed arrays and manifest.")
    if not np.array_equal(array_group, manifest["group_id"].astype(str).to_numpy()):
        raise ValueError("Group values disagree between processed arrays and manifest.")

    target_frame = manifest[["sample_id"]].merge(
        targets[["sample_id", *TARGET_NAMES]], on="sample_id", how="left", validate="one_to_one"
    )
    if target_frame[TARGET_NAMES].isna().any().any():
        raise ValueError("One or more processed examples lack target values.")
    if not np.allclose(theta, target_frame[TARGET_NAMES].to_numpy(dtype=float), rtol=0, atol=1e-12):
        raise ValueError("theta does not match targets.csv in the documented target order.")

    group_split_counts = manifest.groupby("group_id")["split"].nunique()
    if (group_split_counts > 1).any():
        leaking = group_split_counts[group_split_counts > 1].index.tolist()
        raise ValueError(f"Source groups cross split boundaries: {leaking}")

    expected_window = int(settings.get("final_window_seconds", WINDOW_SECONDS) * settings.get("sample_rate_hz", SAMPLE_RATE_HZ))
    if X.shape[1] != expected_window or not (manifest["n_samples"] == expected_window).all():
        raise ValueError(f"Expected {expected_window} samples per example, observed X={X.shape}.")

    metadata = manifest.drop(columns=[name for name in TARGET_NAMES if name in manifest], errors="ignore").merge(
        target_frame, on="sample_id", how="left", validate="one_to_one"
    )
    return {"X": X, "theta": theta, "metadata": metadata, "settings": settings}

## Choose the analysis mode

If the canonical handoff exists, this cell loads and audits it. Otherwise, it creates a small deterministic synthetic dataset solely to exercise every EDA code path. Synthetic output is labeled in figure titles and is never saved as a project artifact.

In [ ]:
def build_synthetic_smoke_dataset(seed=RANDOM_SEED, n_o3a=48, n_o4a=24):
    rng = np.random.default_rng(seed)
    n_examples = n_o3a + n_o4a
    runs = np.array(["O3a"] * n_o3a + ["O4a"] * n_o4a)
    splits = np.array(
        ["train"] * 32 + ["validation"] * 8 + ["O3a_test"] * 8 + ["O4a_test"] * n_o4a
    )
    group_ids = np.array(
        [f"O3a_g{1 + min(i // 8, 5)}" for i in range(n_o3a)]
        + [f"O4a_g{1 + min(i // 12, 1)}" for i in range(n_o4a)]
    )
    noise_ids = np.array([f"{group}_noise" for group in group_ids])
    sample_ids = np.array([f"synthetic_{i:04d}" for i in range(n_examples)])

    chirp_mass = rng.uniform(18, 42, n_examples)
    mass_ratio = rng.uniform(0.5, 1.0, n_examples)
    chi_eff = np.clip(rng.normal(0, 0.22, n_examples), -0.6, 0.6)
    distance = rng.uniform(350, 1200, n_examples)
    theta = np.column_stack([chirp_mass, mass_ratio, chi_eff, distance])

    n_samples = int(SAMPLE_RATE_HZ * WINDOW_SECONDS)
    time_sec = np.arange(n_samples) / SAMPLE_RATE_HZ - WINDOW_SECONDS / 2
    before_merger = time_sec <= 0
    X = np.empty((n_examples, n_samples), dtype=np.float64)
    snr = np.empty(n_examples)
    for index in range(n_examples):
        duration_scale = np.clip(32 / chirp_mass[index], 0.65, 1.5)
        phase_time = np.clip((time_sec + 2) / (2 * duration_scale), 0, 1)
        instantaneous_phase = 2 * np.pi * (35 * (time_sec + 2) + 95 * phase_time**3)
        envelope = np.where(before_merger, np.exp(1.7 * time_sec), 0.0)
        amplitude = 8.0 * (600 / distance[index]) * (chirp_mass[index] / 30) ** (5 / 6)
        waveform = amplitude * envelope * np.sin(instantaneous_phase + 0.5 * chi_eff[index])
        white_noise = rng.normal(0, 1.0 if runs[index] == "O3a" else 1.12, n_samples)
        if runs[index] == "O4a":
            white_noise += 0.10 * np.sin(2 * np.pi * 180 * time_sec)
        X[index] = waveform + white_noise
        snr[index] = np.sqrt(np.mean(waveform**2)) / np.std(white_noise) * np.sqrt(n_samples / 32)

    metadata = pd.DataFrame({
        "row_index": np.arange(n_examples),
        "sample_id": sample_ids,
        "noise_id": noise_ids,
        "group_id": group_ids,
        "run": runs,
        "split": splits,
        "n_samples": n_samples,
        "snr": snr,
        **{name: theta[:, column] for column, name in enumerate(TARGET_NAMES)},
    })
    return {"X": X, "theta": theta, "metadata": metadata, "settings": {
        "sample_rate_hz": SAMPLE_RATE_HZ,
        "final_window_seconds": WINDOW_SECONDS,
        "synthetic_smoke_test": True,
    }}


if REAL_DATA_READY:
    dataset = load_part1_handoff(EXPERIMENT_ROOT)
    ANALYSIS_MODE = "REAL PART 1 ARTIFACTS"
    IS_SYNTHETIC = False
else:
    dataset = build_synthetic_smoke_dataset()
    ANALYSIS_MODE = "SYNTHETIC SMOKE TEST — NOT PROJECT EVIDENCE"
    IS_SYNTHETIC = True

X = dataset["X"]
theta = dataset["theta"]
metadata = dataset["metadata"].copy()
settings = dataset["settings"]
sample_rate_hz = int(settings.get("sample_rate_hz", SAMPLE_RATE_HZ))

print("Analysis mode:", ANALYSIS_MODE)
print("X shape:", X.shape)
print("theta shape:", theta.shape)

## Dataset audit and allocation

These tables establish the denominator for every later plot. In real-data mode, they are the first project results to review with Part 1's owner.

In [ ]:
split_counts = (
    metadata.groupby(["run", "split"], dropna=False)
    .agg(examples=("sample_id", "size"), source_groups=("group_id", "nunique"), noise_sources=("noise_id", "nunique"))
    .reset_index()
)
target_summary = metadata.groupby("run")[TARGET_NAMES].agg(["count", "mean", "std", "min", "median", "max"])

print("Analysis mode:", ANALYSIS_MODE)
print("\nAllocation by run and split:\n", split_counts.to_string(index=False))
print("\nTarget summary by run:\n", target_summary.round(4).to_string())
print("\nFinite strain fraction:", float(np.isfinite(X).mean()))
print("Unique sample IDs:", metadata["sample_id"].is_unique)
print("Maximum splits per group:", int(metadata.groupby("group_id")["split"].nunique().max()))

if IS_SYNTHETIC:
    print("\nPENDING: replace these demonstration counts with the canonical Part 1 handoff before writing findings.")

## Parameter distributions

Normalized step histograms make differently sized O3a and O4a groups comparable. The shared bin edges are computed from the combined displayed population. Once real data are available, check whether each split covers the intended prior and whether apparent run differences are larger than expected from the sample sizes.

In [ ]:
def plot_target_distributions(frame, mode_label):
    figure, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
    for axis, target in zip(axes.flat, TARGET_NAMES):
        values = frame[target].to_numpy(dtype=float)
        bin_edges = np.histogram_bin_edges(values, bins="auto")
        if len(bin_edges) < 6:
            bin_edges = np.linspace(values.min(), values.max(), 8)
        for run_name in [name for name in ["O3a", "O4a"] if name in set(frame["run"])]:
            run_values = frame.loc[frame["run"] == run_name, target]
            axis.hist(
                run_values,
                bins=bin_edges,
                density=True,
                histtype="step",
                linewidth=2,
                color=RUN_COLORS.get(run_name),
                label=f"{run_name} (n={len(run_values)})",
            )
        axis.set_title(target.replace("_", " ").title())
        axis.set_xlabel(TARGET_LABELS[target])
        axis.set_ylabel("Normalized density")
        axis.legend()
    figure.suptitle(f"Target distributions by observing run\n{mode_label}", fontsize=13)
    plt.show()


plot_target_distributions(metadata, ANALYSIS_MODE)

## Target relationships

The heatmap summarizes linear association, while the pairwise scatter plots reveal nonlinear structure, truncation, and gaps in prior coverage. These are relationships among simulated labels, not learned feature importance.

In [ ]:
def plot_target_relationships(frame, mode_label):
    correlation = frame[TARGET_NAMES].corr(method="pearson")
    figure, axis = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
    image = axis.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
    axis.set_xticks(range(len(TARGET_NAMES)), [name.replace("_", " ") for name in TARGET_NAMES], rotation=30, ha="right")
    axis.set_yticks(range(len(TARGET_NAMES)), [name.replace("_", " ") for name in TARGET_NAMES])
    for row in range(len(TARGET_NAMES)):
        for column in range(len(TARGET_NAMES)):
            value = correlation.iloc[row, column]
            axis.text(column, row, f"{value:.2f}", ha="center", va="center", color="white" if abs(value) > 0.55 else "black")
    axis.set_title(f"Pearson target correlations\n{mode_label}")
    figure.colorbar(image, ax=axis, label="Pearson correlation")
    plt.show()

    pairs = [(TARGET_NAMES[i], TARGET_NAMES[j]) for i in range(len(TARGET_NAMES)) for j in range(i + 1, len(TARGET_NAMES))]
    figure, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
    for axis, (x_name, y_name) in zip(axes.flat, pairs):
        for run_name, group in frame.groupby("run"):
            axis.scatter(group[x_name], group[y_name], s=24, alpha=0.65, color=RUN_COLORS.get(run_name), label=run_name)
        axis.set_xlabel(TARGET_LABELS[x_name])
        axis.set_ylabel(TARGET_LABELS[y_name])
    axes.flat[0].legend(loc="best")
    figure.suptitle(f"Pairwise target coverage\n{mode_label}", fontsize=13)
    plt.show()


plot_target_relationships(metadata, ANALYSIS_MODE)

## Representative strain and spectrum plots

Within each run, the displayed example is chosen deterministically as the sample nearest that run's multivariate target median after robust scaling. This avoids selecting an unusually clean or dramatic waveform by eye. Processed-injection plots describe model inputs; they do not isolate detector noise.

In [ ]:
def representative_indices(frame):
    chosen = {}
    for run_name, group in frame.groupby("run"):
        values = group[TARGET_NAMES].to_numpy(dtype=float)
        median = np.median(values, axis=0)
        scale = np.subtract(*np.percentile(values, [75, 25], axis=0))
        scale[scale == 0] = 1.0
        distance_from_center = np.sum(((values - median) / scale) ** 2, axis=1)
        chosen[run_name] = int(group.iloc[np.argmin(distance_from_center)]["row_index"])
    return chosen


def plot_representative_strain(X_values, frame, fs, mode_label):
    selected = representative_indices(frame)
    runs = list(selected)
    figure, axes = plt.subplots(len(runs), 2, figsize=(12, 3.8 * len(runs)), constrained_layout=True, squeeze=False)
    for row, run_name in enumerate(runs):
        index = selected[run_name]
        strain_values = X_values[index]
        time_sec = np.arange(len(strain_values)) / fs - len(strain_values) / (2 * fs)
        frequencies_hz, psd = signal.welch(strain_values, fs=fs, nperseg=min(2048, len(strain_values)), noverlap=min(1024, len(strain_values) // 2))
        record = frame.loc[frame["row_index"] == index].iloc[0]

        axes[row, 0].plot(time_sec, strain_values, color=RUN_COLORS.get(run_name), linewidth=0.8)
        axes[row, 0].axvline(0, color="black", linestyle="--", linewidth=1)
        axes[row, 0].set_title(f"{run_name} representative: {record['sample_id']}")
        axes[row, 0].set_xlabel("Time from merger (s)")
        axes[row, 0].set_ylabel("Processed amplitude")

        valid = frequencies_hz > 0
        axes[row, 1].loglog(frequencies_hz[valid], np.sqrt(psd[valid]), color=RUN_COLORS.get(run_name))
        axes[row, 1].axvspan(35, 350, color=RUN_COLORS.get(run_name), alpha=0.12, label="Analysis band")
        axes[row, 1].set_xlim(20, min(1000, fs / 2))
        axes[row, 1].set_title(f"{run_name} processed-input ASD")
        axes[row, 1].set_xlabel("Frequency (Hz)")
        axes[row, 1].set_ylabel("ASD (output units / sqrt(Hz))")
        axes[row, 1].legend()
    figure.suptitle(f"Representative model-ready inputs\n{mode_label}", fontsize=13)
    plt.show()


plot_representative_strain(X, metadata, sample_rate_hz, ANALYSIS_MODE)

## Processed-input summaries and O3a/O4a comparison

These waveform summaries help detect scale changes and preprocessing anomalies. They are descriptive proxies—not engineered inputs for the posterior model and not proof that an observing-run difference is caused by detector noise.

In [ ]:
def extract_waveform_features(X_values, fs):
    rms = np.sqrt(np.mean(X_values**2, axis=1))
    peak_abs = np.max(np.abs(X_values), axis=1)
    peak_to_rms = peak_abs / np.maximum(rms, np.finfo(float).tiny)
    frequencies_hz = np.fft.rfftfreq(X_values.shape[1], d=1 / fs)
    power = np.abs(np.fft.rfft(X_values, axis=1)) ** 2
    total_power = np.maximum(power.sum(axis=1), np.finfo(float).tiny)
    spectral_centroid = (power * frequencies_hz).sum(axis=1) / total_power

    def band_fraction(low_hz, high_hz):
        mask = (frequencies_hz >= low_hz) & (frequencies_hz < high_hz)
        return power[:, mask].sum(axis=1) / total_power

    return pd.DataFrame({
        "rms": rms,
        "peak_abs": peak_abs,
        "peak_to_rms": peak_to_rms,
        "spectral_centroid_hz": spectral_centroid,
        "power_35_100": band_fraction(35, 100),
        "power_100_200": band_fraction(100, 200),
        "power_200_350": band_fraction(200, 350),
    })


computed_features = extract_waveform_features(X, sample_rate_hz)
overlapping_feature_names = [
    name for name in computed_features.columns if name in metadata.columns
]
for feature_name in overlapping_feature_names:
    manifest_values = metadata[feature_name].to_numpy(dtype=float)
    recomputed_values = computed_features[feature_name].to_numpy(dtype=float)
    if not np.allclose(manifest_values, recomputed_values, rtol=1e-10, atol=1e-12):
        raise ValueError(
            f"Recomputed {feature_name} does not match processed_manifest.csv."
        )

# Avoid duplicate labels when Role 3 already exported summary columns. The EDA
# uses one validated, freshly recomputed copy of each waveform feature.
waveform_features = metadata.drop(
    columns=overlapping_feature_names, errors="ignore"
).reset_index(drop=True)
waveform_features = pd.concat([waveform_features, computed_features], axis=1)
FEATURE_NAMES = ["rms", "peak_abs", "peak_to_rms", "spectral_centroid_hz", "power_35_100", "power_100_200", "power_200_350"]

feature_summary = waveform_features.groupby("run")[FEATURE_NAMES].agg(["count", "median", "mean", "std"])
print(feature_summary.round(5).to_string())

plot_features = ["rms", "peak_to_rms", "spectral_centroid_hz", "power_35_100", "power_100_200", "power_200_350"]
figure, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
run_order = [run for run in ["O3a", "O4a"] if run in set(waveform_features["run"])]
for axis, feature in zip(axes.flat, plot_features):
    values = [waveform_features.loc[waveform_features["run"] == run, feature] for run in run_order]
    box = axis.boxplot(values, patch_artist=True, showfliers=True)
    axis.set_xticks(np.arange(1, len(run_order) + 1), run_order)
    for patch, run in zip(box["boxes"], run_order):
        patch.set_facecolor(RUN_COLORS.get(run, "#999999"))
        patch.set_alpha(0.55)
    axis.set_title(feature.replace("_", " ").title())
    axis.set_xlabel("Observing run")
    axis.set_ylabel("Value")
figure.suptitle(f"Processed-input summaries by run\n{ANALYSIS_MODE}", fontsize=13)
plt.show()

## Noise-only O3a/O4a ASD comparison

This is the preferred detector-drift view because it does not mix different injected parameters into the comparison. It activates only when Part 1 provides both the accepted noise manifest and Role 3 PSD files.

In [ ]:
def plot_noise_asd_comparison(noise_manifest_path, psd_dir):
    if not noise_manifest_path.is_file() or not psd_dir.is_dir():
        print("PENDING: noise-only ASD comparison requires noise_manifest.csv and preprocessing/psd/*.npz.")
        return False

    noise_manifest = pd.read_csv(noise_manifest_path)
    require_columns(noise_manifest, ["noise_id", "run"], "noise_manifest.csv")
    if "quality_status" in noise_manifest:
        noise_manifest = noise_manifest[noise_manifest["quality_status"].astype(str).str.lower() == "accepted"]

    spectra = []
    for row in noise_manifest.itertuples(index=False):
        path = psd_dir / f"{row.noise_id}.npz"
        if not path.is_file():
            continue
        with np.load(path, allow_pickle=False) as values:
            spectra.append({
                "noise_id": str(row.noise_id),
                "run": str(row.run),
                "frequency_hz": values["frequency_hz"].astype(float),
                "asd": np.sqrt(values["psd_strain2_per_hz"].astype(float)),
            })
    if not spectra:
        print("PENDING: no PSD files match accepted noise IDs.")
        return False

    common_frequency = spectra[0]["frequency_hz"]
    figure, axis = plt.subplots(figsize=(9, 5), constrained_layout=True)
    for run_name in sorted({record["run"] for record in spectra}):
        run_spectra = [record for record in spectra if record["run"] == run_name]
        matrix = np.stack([np.interp(common_frequency, record["frequency_hz"], record["asd"]) for record in run_spectra])
        median = np.median(matrix, axis=0)
        low, high = np.percentile(matrix, [25, 75], axis=0)
        valid = (common_frequency >= 20) & (common_frequency <= 1000)
        axis.loglog(common_frequency[valid], median[valid], color=RUN_COLORS.get(run_name), label=f"{run_name} median (n={len(run_spectra)})")
        if len(run_spectra) > 1:
            axis.fill_between(common_frequency[valid], low[valid], high[valid], color=RUN_COLORS.get(run_name), alpha=0.18, label=f"{run_name} IQR")
    axis.axvspan(35, 350, color="gray", alpha=0.12, label="Analysis band")
    axis.set_title("Noise-only amplitude spectral density by observing run")
    axis.set_xlabel("Frequency (Hz)")
    axis.set_ylabel("ASD (strain / sqrt(Hz))")
    axis.legend()
    plt.show()
    return True


NOISE_ASD_READY = plot_noise_asd_comparison(
    ARTIFACT_PATHS["noise manifest"], PREPROCESSING_DIR / "psd"
)

## Injection-selection diagnostics

If the injection manifest is available, this section checks whether SNR selection changes the apparent target population and whether distance behaves as expected. It is descriptive only; selection effects must be considered before interpreting O3a/O4a target differences.

In [ ]:
if ARTIFACT_PATHS["injection manifest"].is_file():
    injection_manifest = pd.read_csv(ARTIFACT_PATHS["injection manifest"])
    require_columns(injection_manifest, ["sample_id", "run", "split", "snr"], "injection_manifest.csv")
    selection_frame = metadata[["sample_id", *TARGET_NAMES]].merge(
        injection_manifest[["sample_id", "run", "split", "snr"]], on="sample_id", how="inner", validate="one_to_one"
    )
elif "snr" in metadata:
    selection_frame = metadata[["sample_id", "run", "split", "snr", *TARGET_NAMES]].copy()
else:
    selection_frame = pd.DataFrame()

if selection_frame.empty:
    print("PENDING: SNR diagnostics require injection_manifest.csv with retained sample IDs.")
else:
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    for run_name, group in selection_frame.groupby("run"):
        axes[0].hist(group["snr"], bins="auto", histtype="step", linewidth=2, color=RUN_COLORS.get(run_name), label=f"{run_name} (n={len(group)})")
        axes[1].scatter(group["distance"], group["snr"], s=25, alpha=0.65, color=RUN_COLORS.get(run_name), label=run_name)
    axes[0].set(title="Retained optimal-SNR distribution", xlabel="Optimal SNR (dimensionless)", ylabel="Examples")
    axes[1].set(title="Distance and retained optimal SNR", xlabel="Luminosity distance (Mpc)", ylabel="Optimal SNR (dimensionless)")
    axes[0].legend()
    axes[1].legend()
    figure.suptitle(ANALYSIS_MODE)
    plt.show()

## Completion status and handoff

The checks below distinguish implemented code from available project evidence. A green implementation check does not imply that the corresponding scientific conclusion is ready.

In [ ]:
completion = pd.DataFrame([
    ("Part 1 processed arrays loaded", "complete" if REAL_DATA_READY else "pending", "Canonical processed.npz and manifests"),
    ("Artifact schema and leakage audit", "complete" if REAL_DATA_READY else "smoke-tested", "IDs, shapes, finite values, target order, group splits"),
    ("Parameter distributions", "complete" if REAL_DATA_READY else "smoke-tested", "Four targets compared by run"),
    ("Target relationships", "complete" if REAL_DATA_READY else "smoke-tested", "Correlation heatmap and pairwise coverage"),
    ("Representative strain plots", "complete" if REAL_DATA_READY else "smoke-tested", "Deterministic central examples by run"),
    ("Processed-input run comparison", "complete" if REAL_DATA_READY else "smoke-tested", "Waveform summary distributions"),
    ("Noise-only O3a/O4a ASD", "complete" if NOISE_ASD_READY else "pending", "Accepted noise PSD files"),
    ("Project findings written", "pending" if IS_SYNTHETIC else "ready for interpretation", "Must be based on real executed outputs"),
], columns=["item", "status", "evidence_required"])
print(completion.to_string(index=False))

## Findings template — complete only after the real-data run

Do **not** replace the placeholders below with synthetic values.

1. **Dataset and split:** The canonical handoff contains **[N]** examples with **[counts by split]**. Group-disjointness **[passed/failed]**.
2. **Parameter coverage:** The four targets **[do/do not]** cover the intended prior across development and evaluation splits. The most important imbalance is **[finding]**.
3. **Target relationships:** The strongest observed label association is **[pair and correlation]**. This is **[expected/unexpected]** because **[reason]**.
4. **Strain behavior:** Representative processed inputs show **[finding]**, while the distribution of RMS/peak/spectral summaries shows **[finding]**.
5. **O3a versus O4a:** Noise-only ASD comparison shows **[frequency-dependent finding]**. Processed-input differences are interpreted cautiously because **[target/SNR confounder]**.
6. **Modeling handoff:** Part 3 should use **[exact train/validation/test counts]**, retain the documented target order, and monitor **[identified risk]**.

## Takeaways

- The EDA implementation is ready to consume the agreed Part 1 contract.
- Until the real artifacts exist locally, all displayed distributions and relationships are synthetic smoke tests rather than project findings.
- The final rerun must review allocation, leakage, target coverage, representative strain, SNR selection, and noise-only O3a/O4a ASD before Part 3 begins model comparison.
- Any conclusion about observing-run drift should prioritize noise-only spectra and explicitly separate detector behavior from simulated-parameter and selection effects.